In [1]:
import openai

In [2]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [ ]:
llm_model = 'gpt-5-nano-2025-08-07'

In [89]:
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router import MultiPromptChain
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain, ConversationChain
from langchain.memory import ConversationTokenBufferMemory, ConversationBufferWindowMemory

In [90]:
guitar_memory = ConversationBufferWindowMemory(k=2, memory_key="history")

In [91]:
guitar_template = """You are a very skilled guitarist and good guitar instructor. \
You are great at answering questions about learning and playing the guitar in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

History:
{history}

Here is a question:
{input}"""


photography_template = """You are a very good photographer. \
You are great at answering questions related to photography and how to get good shots. \
You are so good because you are able to undertand composition, color grading, 
and all the camera setups like shutter speed, aperture, etc. needed for a good photo. 
Give concise answers for a beginner.

Here is a question:
{input}"""

In [92]:
prompt_infos = [
    {
        "name":"guitar",
        "description":"Good for answering questions about playing the guitar",
        "prompt_template": guitar_template
    },
    {
        "name":"photography",
        "description":"Good for answering questions about photography",
        "prompt_template": photography_template
    }
]

In [93]:
llm = ChatOpenAI(model=llm_model, temperature=0.4)

In [94]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    if name == "guitar":
        prompt = PromptTemplate(
            template=prompt_template,
            input_variables=["history","input"]
        )
        chain = LLMChain(llm=llm, prompt=prompt, memory=guitar_memory)
    else:
        prompt = PromptTemplate(
            template=prompt_template,
            input_variables=["input"]
        )
        chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}:{p['description']}" for p in prompt_infos]
destinations_str = '\n'.join(destinations)

In [95]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [96]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [97]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

router_prompt = PromptTemplate(
    template = router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(llm=llm, prompt=router_prompt)

In [98]:
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True
)

In [99]:
chain.run("What are the strings of the guitar called?")



> Entering new MultiPromptChain chain...
guitar: {'input': 'What are the names of the strings on a guitar?'}
> Finished chain.


'The strings on a standard guitar are usually referred to by their note names, from the thickest (6th string) to the thinnest (1st string). Here they are:\n\n1. **6th string** - E (low E)\n2. **5th string** - A\n3. **4th string** - D\n4. **3rd string** - G\n5. **2nd string** - B\n6. **1st string** - E (high E)\n\nA common mnemonic to remember the order is "Eddie Ate Dynamite, Good Bye Eddie."'

In [100]:
chain.run("What is shutter speed? Answer in 20 words")



> Entering new MultiPromptChain chain...
photography: {'input': 'What is shutter speed? Please provide a brief explanation in 20 words.'}
> Finished chain.


"Shutter speed is the duration the camera's shutter is open, controlling light exposure and motion blur in photos."

In [101]:
chain.run("I can only play C, A, D, E chords. Give me 3 popular songs I can play that only use these chords.")



> Entering new MultiPromptChain chain...
guitar: {'input': 'I can only play C, A, D, E chords. Can you give me 3 popular songs that only use these chords?'}
> Finished chain.


'Sure! Here are three popular songs that primarily use the chords C, A, D, and E:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - The main progression can be simplified to fit within these chords.\n  \n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song primarily uses the chords C, A, and D.\n\n3. **"Horse with No Name" by America** - While it uses a slightly different structure, you can adapt it to fit within the chords you know.\n\nThese songs should be fun to play with the chords you already know!'

In [102]:
guitar_memory.load_memory_variables({})

{'history': 'Human: What are the names of the strings on a guitar?\nAI: The strings on a standard guitar are usually referred to by their note names, from the thickest (6th string) to the thinnest (1st string). Here they are:\n\n1. **6th string** - E (low E)\n2. **5th string** - A\n3. **4th string** - D\n4. **3rd string** - G\n5. **2nd string** - B\n6. **1st string** - E (high E)\n\nA common mnemonic to remember the order is "Eddie Ate Dynamite, Good Bye Eddie."\nHuman: I can only play C, A, D, E chords. Can you give me 3 popular songs that only use these chords?\nAI: Sure! Here are three popular songs that primarily use the chords C, A, D, and E:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - The main progression can be simplified to fit within these chords.\n  \n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song primarily uses the chords C, A, and D.\n\n3. **"Horse with No Name" by America** - While it uses a slightly different structure, you can adapt it to fit within the

In [103]:
chain.run("What genre are these 3 guitar songs from?")



> Entering new MultiPromptChain chain...
guitar: {'input': 'What genre are these 3 guitar songs from?'}
> Finished chain.


'The three songs mentioned span different genres:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - This song is primarily classified as Southern rock, with elements of country and blues.\n\n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song is in the country genre.\n\n3. **"Horse with No Name" by America** - This song is often categorized as folk rock or soft rock.\n\nSo, you have a mix of Southern rock, country, and folk rock!'

In [104]:
guitar_memory.load_memory_variables({})

{'history': 'Human: I can only play C, A, D, E chords. Can you give me 3 popular songs that only use these chords?\nAI: Sure! Here are three popular songs that primarily use the chords C, A, D, and E:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - The main progression can be simplified to fit within these chords.\n  \n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song primarily uses the chords C, A, and D.\n\n3. **"Horse with No Name" by America** - While it uses a slightly different structure, you can adapt it to fit within the chords you know.\n\nThese songs should be fun to play with the chords you already know!\nHuman: What genre are these 3 guitar songs from?\nAI: The three songs mentioned span different genres:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - This song is primarily classified as Southern rock, with elements of country and blues.\n\n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song is in the country genre.\n\n3. **"Horse with No Name" by Ameri

In [105]:
chain.run("Tell me one more guitar song that can be played with the chords I know")



> Entering new MultiPromptChain chain...
guitar: {'input': 'Can you suggest one more guitar song that can be played with the chords I know?'}
> Finished chain.


'Absolutely! Another song you can play with the chords C, A, D, and E is **"Three Little Birds" by Bob Marley**. This song has a simple chord progression and is a great addition to your repertoire. Enjoy playing!'

In [106]:
guitar_memory.load_memory_variables({})

{'history': 'Human: What genre are these 3 guitar songs from?\nAI: The three songs mentioned span different genres:\n\n1. **"Sweet Home Alabama" by Lynyrd Skynyrd** - This song is primarily classified as Southern rock, with elements of country and blues.\n\n2. **"Achy Breaky Heart" by Billy Ray Cyrus** - This song is in the country genre.\n\n3. **"Horse with No Name" by America** - This song is often categorized as folk rock or soft rock.\n\nSo, you have a mix of Southern rock, country, and folk rock!\nHuman: Can you suggest one more guitar song that can be played with the chords I know?\nAI: Absolutely! Another song you can play with the chords C, A, D, and E is **"Three Little Birds" by Bob Marley**. This song has a simple chord progression and is a great addition to your repertoire. Enjoy playing!'}

In [107]:
chain.run("What guitar chords do I know?")



> Entering new MultiPromptChain chain...
guitar: {'input': 'What guitar chords do I know?'}
> Finished chain.


"I don't have access to your specific knowledge or the chords you know. However, if you tell me which chords you are familiar with, I can suggest songs or exercises that use those chords!"